# S&P 500 — Daily + Monthly extraction (Python / WRDS)

Port direct de `src/sas/data_extraction_sp500.sas` vers Python. Mêmes sources CRSP, mêmes outputs (sans GVKEY — l'ESG sera traité dans un script séparé).

## Setup (à faire une fois)

**1. Installer les dépendances** dans le venv du projet :

```bash
pip install wrds pandas pyarrow ipykernel
```

**2. Première connexion à WRDS** : la cellule de connexion ci-dessous va te demander ton mot de passe WRDS au premier run, puis te proposer de créer un fichier `pgpass` (Windows : `%APPDATA%\postgresql\pgpass.conf`). Accepte. Les runs suivants seront silencieux.

**3. Outputs** : 3 fichiers Parquet écrits dans `data/cache/` (gitignored par défaut grâce à la règle `data/` ? — à vérifier, sinon ajouter).

In [1]:
import wrds
import pandas as pd
import numpy as np
from pathlib import Path

## 0) Connexion WRDS

Au premier appel : prompt password puis offre de créer `pgpass`. Dis `y`.

In [11]:
WRDS_USERNAME = 'aadmberrada'  # change si nécessaire
db = wrds.Connection(wrds_username=WRDS_USERNAME)

Loading library list...
Done


## Configuration

In [49]:
START_DATE = '2020-01-01'
END_DATE   = '2026-06-30'

OUT_DIR = Path('../data/wrds')
OUT_DIR.mkdir(parents=True, exist_ok=True)

## 1) Jours de bourse (CRSP DSI)

In [50]:
trading_days = db.raw_sql(f"""
    SELECT DISTINCT date
    FROM crsp.dsi
    WHERE date BETWEEN '{START_DATE}' AND '{END_DATE}'
    ORDER BY date
""", date_cols=['date'])

trading_days['date'] = pd.to_datetime(trading_days['date'])
print(f"Trading days: {len(trading_days):,}")
print(f'The trading days span from {trading_days['date'].min()} to {trading_days['date'].max()}')

Trading days: 1,258
The trading days span from 2020-01-02 00:00:00 to 2024-12-31 00:00:00


In [26]:
trading_days.tail()

,date
1253,2024-12-24
1254,2024-12-26
1255,2024-12-27
1256,2024-12-30
1257,2024-12-31


## 2) Membership S&P 500 par jour (DSP500LIST_V2)

Point-in-time, pas de survivorship bias.

In [22]:
sp500_membership = db.raw_sql(f"""
    SELECT t.date,
           m.permno,
           m.mbrstartdt AS mbr_startdt,
           m.mbrenddt   AS mbr_enddt
    FROM (SELECT DISTINCT date FROM crsp.dsi
          WHERE date BETWEEN '{START_DATE}' AND '{END_DATE}') t
    INNER JOIN crsp.dsp500list_v2 m
      ON t.date BETWEEN m.mbrstartdt AND COALESCE(m.mbrenddt, '9999-12-31'::date)
    ORDER BY t.date, m.permno
""", date_cols=['date', 'mbr_startdt', 'mbr_enddt'])

print(f"Membership rows: {len(sp500_membership):,}")
sp500_membership.head()

Membership rows: 633,989


,date,permno,mbr_startdt,mbr_enddt
0,2020-01-02,10104,1989-08-03,2025-12-31
1,2020-01-02,10107,1994-06-07,2025-12-31
2,2020-01-02,10138,1999-10-13,2025-12-31
3,2020-01-02,10145,1925-12-31,2025-12-31
4,2020-01-02,10516,1981-07-30,2025-12-31


## 3) Ticker / nom valides le jour J (STOCKNAMES) → constituents

Filtre `shrcd in (10, 11)`, dédupe à 1 ligne par `(date, permno)` en gardant le `namedt` le plus récent.

In [23]:
sp500_day_raw = db.raw_sql(f"""
    SELECT m.date,
           m.permno,
           m.mbrstartdt AS mbr_startdt,
           m.mbrenddt   AS mbr_enddt,
           s.permco,
           s.ticker,
           s.comnam,
           s.cusip,
           s.shrcd,
           s.shrcls,
           s.namedt,
           COALESCE(s.nameenddt, '9999-12-31'::date) AS nameenddt
    FROM (
        SELECT t.date, m.permno, m.mbrstartdt, m.mbrenddt
        FROM (SELECT DISTINCT date FROM crsp.dsi
              WHERE date BETWEEN '{START_DATE}' AND '{END_DATE}') t
        INNER JOIN crsp.dsp500list_v2 m
          ON t.date BETWEEN m.mbrstartdt AND COALESCE(m.mbrenddt, '9999-12-31'::date)
    ) m
    LEFT JOIN crsp.stocknames s
      ON  m.permno = s.permno
      AND m.date BETWEEN s.namedt AND COALESCE(s.nameenddt, '9999-12-31'::date)
      AND s.shrcd IN (10, 11)
""", date_cols=['date', 'mbr_startdt', 'mbr_enddt', 'namedt', 'nameenddt'])

# Dedupe : 1 ligne par (date, permno), prendre le namedt le plus récent
sp500_day = (sp500_day_raw
             .sort_values(['date', 'permno', 'namedt', 'nameenddt'],
                          ascending=[True, True, False, False])
             .drop_duplicates(subset=['date', 'permno'], keep='first')
             .reset_index(drop=True))

print(f"Constituents daily rows: {len(sp500_day):,}")
sp500_day.head()

Constituents daily rows: 633,989


,date,permno,mbr_startdt,mbr_enddt,permco,ticker,comnam,cusip,shrcd,shrcls,namedt,nameenddt
0,2020-01-02,10104,1989-08-03,2025-12-31,8045,ORCL,ORACLE CORP,68389X10,11,<NA>,2013-07-15,2023-06-26
1,2020-01-02,10107,1994-06-07,2025-12-31,8048,MSFT,MICROSOFT CORP,59491810,11,<NA>,1986-03-13,2024-12-31
2,2020-01-02,10138,1999-10-13,2025-12-31,8087,TROW,T ROWE PRICE GROUP INC,74144T10,11,<NA>,2000-12-29,2024-12-31
3,2020-01-02,10145,1925-12-31,2025-12-31,22168,HON,HONEYWELL INTERNATIONAL INC,43851610,11,<NA>,2017-01-30,2021-04-05
4,2020-01-02,10516,1981-07-30,2025-12-31,20207,ADM,ARCHER DANIELS MIDLAND CO,03948310,11,<NA>,2012-10-31,2024-12-31


In [24]:
# Persist constituents (sans GVKEY)
sp500_constituents_daily = sp500_day.assign(
    ticker = sp500_day['ticker'].str.upper(),
    comnam = sp500_day['comnam'].str.upper(),
    cusip  = sp500_day['cusip'].str.upper(),
).sort_values(['date', 'permno']).reset_index(drop=True)

sp500_constituents_daily.to_parquet(OUT_DIR / 'sp500_constituents_daily.parquet')
print(f"Saved → {OUT_DIR / 'sp500_constituents_daily.parquet'}")
print(f"Rows: {len(sp500_constituents_daily):,}")
print(f"PERMNO uniques: {sp500_constituents_daily['permno'].nunique():,}")
print(f"Days uniques: {sp500_constituents_daily['date'].nunique():,}")

Saved → ..\data\cache\sp500_constituents_daily.parquet
Rows: 633,989
PERMNO uniques: 594
Days uniques: 1,258


## 4) Prix / Retours / Liquidité (CRSP DSF)

Pull restreint à l'univers S&P 500, **chunké par année** pour éviter une requête trop large.

Si tu veux d'abord tester sur une fenêtre courte, change `START_DATE` / `END_DATE` plus haut et relance.

In [25]:
univ_permno = tuple(int(p) for p in sp500_constituents_daily['permno'].unique())
print(f"Univers PERMNO total: {len(univ_permno):,}")

years = range(int(START_DATE[:4]), int(END_DATE[:4]) + 1)

dsf_chunks = []
for y in years:
    y_start = max(f'{y}-01-01', START_DATE)
    y_end   = min(f'{y}-12-31', END_DATE)

    chunk = db.raw_sql(f"""
        SELECT date, permno, permco,
               prc, retx, ret, vol, shrout, bidlo, askhi, cfacpr, cfacshr,
               hexcd, hsiccd, issuno,
               cusip AS cusip_hdr,
               bid, ask, openprc, numtrd
        FROM crsp.dsf
        WHERE date BETWEEN '{y_start}' AND '{y_end}'
          AND permno IN {univ_permno}
    """, date_cols=['date'])
    dsf_chunks.append(chunk)
    print(f"  {y}: {len(chunk):,} rows")

dsf_core = pd.concat(dsf_chunks, ignore_index=True)
del dsf_chunks
print(f"Total DSF rows: {len(dsf_core):,}")

Univers PERMNO total: 594
  2020: 144,528 rows
  2021: 143,676 rows
  2022: 141,376 rows
  2023: 139,913 rows
  2024: 141,235 rows
  2025: 0 rows
  2026: 0 rows
Total DSF rows: 710,728


C:\Users\abdou\AppData\Local\Temp\ipykernel_4072\966681435.py:24: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  dsf_core = pd.concat(dsf_chunks, ignore_index=True)


## 5) Distributions & Splits (CRSP DSEDIST)

Agrégation au jour `(permno, exdt)` ; produits multiplicatifs des facteurs via somme des logs.

In [27]:
dist_raw = db.raw_sql(f"""
    SELECT permno, exdt AS date,
           divamt, facpr, facshr,
           distcd, paydt, rcrddt
    FROM crsp.dsedist
    WHERE exdt BETWEEN '{START_DATE}' AND '{END_DATE}'
""", date_cols=['date', 'paydt', 'rcrddt'])

dist_daily_agg = (dist_raw
    .assign(
        log_facpr_pos  = np.where(dist_raw['facpr']  > 0, np.log(dist_raw['facpr'].clip(lower=1e-30)),  0.0),
        log_facshr_pos = np.where(dist_raw['facshr'] > 0, np.log(dist_raw['facshr'].clip(lower=1e-30)), 0.0),
        divamt_filled  = dist_raw['divamt'].fillna(0.0),
    )
    .groupby(['permno', 'date'], as_index=False)
    .agg(
        divamt_sum       = ('divamt_filled', 'sum'),
        facpr_log_sum    = ('log_facpr_pos', 'sum'),
        facshr_log_sum   = ('log_facshr_pos', 'sum'),
        dist_event_count = ('distcd', 'size'),
        distcd_min       = ('distcd', 'min'),
        distcd_max       = ('distcd', 'max'),
        paydt_min        = ('paydt', 'min'),
        paydt_max        = ('paydt', 'max'),
        rcrddt_min       = ('rcrddt', 'min'),
        rcrddt_max       = ('rcrddt', 'max'),
    )
    .assign(
        facpr_prod  = lambda d: np.exp(d['facpr_log_sum']),
        facshr_prod = lambda d: np.exp(d['facshr_log_sum']),
    )
    .drop(columns=['facpr_log_sum', 'facshr_log_sum']))

print(f"Distribution events aggregated: {len(dist_daily_agg):,}")
dist_daily_agg.head()

Distribution events aggregated: 131,786


,permno,date,divamt_sum,dist_event_count,distcd_min,distcd_max,paydt_min,paydt_max,rcrddt_min,rcrddt_max,facpr_prod,facshr_prod
0,10026,2020-03-16,0.575,1,1232,1232,2020-04-08,2020-04-08,2020-03-17,2020-03-17,1.0,1.0
1,10026,2020-06-12,0.575,1,1232,1232,2020-07-07,2020-07-07,2020-06-15,2020-06-15,1.0,1.0
2,10026,2020-09-28,0.575,1,1232,1232,2020-10-13,2020-10-13,2020-09-29,2020-09-29,1.0,1.0
3,10026,2020-12-18,0.575,1,1232,1232,2021-01-12,2021-01-12,2020-12-21,2020-12-21,1.0,1.0
4,10026,2021-03-19,0.575,1,1232,1232,2021-04-13,2021-04-13,2021-03-22,2021-03-22,1.0,1.0


## 6) Delistings (CRSP DSEDELIST)

In [28]:
delist_daily = db.raw_sql(f"""
    SELECT permno, dlstdt AS date, dlstcd, dlprc, dlret
    FROM crsp.dsedelist
    WHERE dlstdt BETWEEN '{START_DATE}' AND '{END_DATE}'
    ORDER BY dlstdt, permno
""", date_cols=['date'])

print(f"Delistings: {len(delist_daily):,}")

Delistings: 12,904


## 7) Indices de marché (CRSP DSI core)

In [29]:
dsi_core = db.raw_sql(f"""
    SELECT date,
           vwretd, ewretd,
           vwretx, ewretx,
           sprtrn, spindx,
           totcnt, usdcnt, totval, usdval
    FROM crsp.dsi
    WHERE date BETWEEN '{START_DATE}' AND '{END_DATE}'
""", date_cols=['date'])

print(f"DSI rows: {len(dsi_core):,}")

DSI rows: 1,258


## 8) Table finale journalière `sp500_daily_full`

Left-joins constituents + DSF + DIST + DELIST + DSI ; ajoute les dérivés `me`, `turnover`, `dollar_vol`, `ret_combined`.

In [31]:
univ = sp500_constituents_daily[[
    'date', 'permno', 'permco', 'mbr_startdt', 'mbr_enddt',
    'ticker', 'comnam', 'cusip', 'shrcd', 'shrcls', 'namedt', 'nameenddt'
]]

# DSF sans permco (déjà dans univ) pour éviter la collision
dsf_for_join = dsf_core.drop(columns=['permco'])

panel = (univ
    .merge(dsf_for_join,    on=['date', 'permno'], how='left')
    .merge(dist_daily_agg,  on=['date', 'permno'], how='left')
    .merge(delist_daily,    on=['date', 'permno'], how='left')
    .merge(dsi_core,        on='date',             how='left')
)

# WRDS renvoie parfois des dtypes nullable (Int64/Float64) -> on force float64
# pour éviter "boolean value of NA is ambiguous" dans np.where et np.log1p plus loin.
numeric_cols = [
    'prc', 'retx', 'ret', 'vol', 'shrout', 'bidlo', 'askhi', 'cfacpr', 'cfacshr',
    'hexcd', 'hsiccd', 'issuno', 'bid', 'ask', 'openprc', 'numtrd',
    'divamt_sum', 'facpr_prod', 'facshr_prod', 'dist_event_count',
    'distcd_min', 'distcd_max',
    'dlstcd', 'dlprc', 'dlret',
    'vwretd', 'ewretd', 'vwretx', 'ewretx', 'sprtrn', 'spindx',
    'totcnt', 'usdcnt', 'totval', 'usdval',
]
for col in numeric_cols:
    if col in panel.columns:
        panel[col] = pd.to_numeric(panel[col], errors='coerce').astype('float64')

# Dérivés
panel['me']         = panel['prc'].abs() * panel['shrout']
panel['turnover']   = np.where(panel['shrout'] > 0, panel['vol'] / panel['shrout'], np.nan)
panel['dollar_vol'] = panel['prc'].abs() * panel['vol']

# RET_COMBINED : splice du delisting return dans le retour total
panel['ret_combined'] = np.where(
    panel['dlret'].notna(),
    (1 + panel['ret'].fillna(0)) * (1 + panel['dlret']) - 1,
    panel['ret']
)

sp500_daily_full = panel.sort_values(['date', 'permno']).reset_index(drop=True)
sp500_daily_full.to_parquet(OUT_DIR / 'sp500_daily_full.parquet')
print(f"Saved → {OUT_DIR / 'sp500_daily_full.parquet'}")
print(f"Rows: {len(sp500_daily_full):,}, Cols: {sp500_daily_full.shape[1]}")

Saved → ..\data\cache\sp500_daily_full.parquet
Rows: 633,989, Cols: 56


### Sanity checks (daily)

In [32]:
checks = pd.Series({
    'n_rows':              len(sp500_daily_full),
    'n_unique_keys':       sp500_daily_full[['date', 'permno']].drop_duplicates().shape[0],
    'n_ret':               sp500_daily_full['ret'].notna().sum(),
    'n_dlret':             sp500_daily_full['dlret'].notna().sum(),
    'n_ret_combined':      sp500_daily_full['ret_combined'].notna().sum(),
    'n_missing_price_days': ((sp500_daily_full['prc'].isna()) & (sp500_daily_full['ret'].isna())).sum(),
    'n_div_days':          (sp500_daily_full['divamt_sum'].fillna(0) > 0).sum(),
    'n_vwretx_filled':     sp500_daily_full['vwretx'].notna().sum(),
})
checks

n_rows                  633989
n_unique_keys           633989
n_ret                   633927
n_dlret                     29
n_ret_combined          633927
n_missing_price_days        44
n_div_days                7983
n_vwretx_filled         633989
dtype: int64

## 9) Version mensuelle `sp500_monthly_full`

Règles (identiques à la version SAS) :
- `date` = dernier jour de bourse du mois (EOM par PERMNO).
- Niveaux (PRC, SHROUT, BID, ASK, …) = valeur du dernier jour du mois.
- Flux (VOL, DOLLAR_VOL, NUMTRD, DIVAMT_SUM, …) = sommes mensuelles.
- Retours (RET, RETX, RET_COMBINED, VWRETD, …) = composés via `exp(sum(log(1+r))) - 1`.

In [ ]:
df = sp500_daily_full.copy()
df['mdate'] = (df['date'] + pd.offsets.MonthEnd(0)).dt.normalize()

# 1) Dernier jour observé du mois par PERMNO
last_in_month = (df.groupby(['permno', 'mdate'], as_index=False)['date']
                   .max()
                   .rename(columns={'date': 'last_dt'}))

# 2) Niveaux pris au dernier jour du mois
level_cols = [
    'permco', 'mbr_startdt', 'mbr_enddt',
    'ticker', 'comnam', 'cusip', 'shrcd', 'shrcls', 'namedt', 'nameenddt',
    'prc', 'shrout', 'bidlo', 'askhi', 'openprc',
    'hexcd', 'hsiccd', 'issuno', 'cusip_hdr', 'bid', 'ask',
    'dlstcd', 'dlprc', 'dlret', 'spindx',
]
levels_last = (df.merge(last_in_month, on=['permno', 'mdate'])
                 .query('date == last_dt')
                 [['permno', 'mdate', 'last_dt'] + level_cols])

# 3) Agrégations mensuelles (somme + retours composés) par PERMNO
def compound(s: pd.Series) -> float:
    s = s.dropna()
    return np.expm1(np.log1p(s).sum()) if len(s) else np.nan

agg_month = (df.groupby(['permno', 'mdate']).agg(
    vol              = ('vol',              'sum'),
    dollar_vol       = ('dollar_vol',       'sum'),
    numtrd           = ('numtrd',           'sum'),
    divamt_sum       = ('divamt_sum',       'sum'),
    dist_event_count = ('dist_event_count', 'sum'),
    distcd_min       = ('distcd_min',       'min'),
    distcd_max       = ('distcd_max',       'max'),
    paydt_min        = ('paydt_min',        'min'),
    paydt_max        = ('paydt_max',        'max'),
    rcrddt_min       = ('rcrddt_min',       'min'),
    rcrddt_max       = ('rcrddt_max',       'max'),
    n_trd_days       = ('prc', lambda s: ((s.notna()) | (df.loc[s.index, 'ret'].notna())).sum()),
    ret              = ('ret',              compound),
    retx             = ('retx',             compound),
    ret_combined     = ('ret_combined',     compound),
).reset_index())

# 4) Benchmarks mensuels (mêmes valeurs pour toutes les lignes du mois)
market_month = (df.groupby('mdate').agg(
    vwretd = ('vwretd', compound),
    ewretd = ('ewretd', compound),
    vwretx = ('vwretx', compound),
    ewretx = ('ewretx', compound),
    sprtrn = ('sprtrn', compound),
    totcnt = ('totcnt', 'sum'),
    usdcnt = ('usdcnt', 'sum'),
    totval = ('totval', 'sum'),
    usdval = ('usdval', 'sum'),
).reset_index())

# 5) Assemblage final
sp500_monthly_full = (levels_last
    .merge(agg_month,    on=['permno', 'mdate'], how='left')
    .merge(market_month, on='mdate',             how='left')
    .rename(columns={'mdate': 'date'})
    .sort_values(['date', 'permno'])
    .reset_index(drop=True))

sp500_monthly_full.to_parquet(OUT_DIR / 'sp500_monthly_full.parquet')
print(f"Saved → {OUT_DIR / 'sp500_monthly_full.parquet'}")
print(f"Rows: {len(sp500_monthly_full):,}, Months: {sp500_monthly_full['date'].nunique():,}")

### Sanity checks (monthly)

In [ ]:
monthly_checks = pd.Series({
    'n_rows':         len(sp500_monthly_full),
    'n_months':       sp500_monthly_full['date'].nunique(),
    'n_permno':       sp500_monthly_full['permno'].nunique(),
    'n_unique_keys':  sp500_monthly_full[['date', 'permno']].drop_duplicates().shape[0],
})
monthly_checks

## 10) Disconnect

In [ ]:
db.close()